In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [4]:
print(tf.__version__)

2.21.0


In [5]:
df = pd.read_csv("../dataset/spam_processed.csv")

# Null values error fixed by filling with empty string
df["processed_message"] = (df["processed_message"].fillna(""))

df.head()

,label,message,processed_message
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah dont think goe usf live around though


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5169 entries, 0 to 5168
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   label              5169 non-null   int64
 1   message            5169 non-null   str  
 2   processed_message  5169 non-null   str  
dtypes: int64(1), str(2)
memory usage: 121.3 KB


In [7]:
X = df["processed_message"]

y = df["label"]

In [8]:
X.head()

0    go jurong point crazi avail bugi n great world...
1                                ok lar joke wif u oni
2    free entri wkli comp win fa cup final tkt st m...
3                  u dun say earli hor u c alreadi say
4            nah dont think goe usf live around though
Name: processed_message, dtype: str

In [9]:
tokenizer = Tokenizer(num_words=5000,oov_token="<OOV>")

In [10]:
tokenizer.fit_on_texts(X)

In [11]:
len(tokenizer.word_index)

7069

In [12]:
list(tokenizer.word_index.items())[:20]

[('<OOV>', 1),
 ('u', 2),
 ('call', 3),
 ('im', 4),
 ('go', 5),
 ('get', 6),
 ('ur', 7),
 ('come', 8),
 ('dont', 9),
 ('ltgt', 10),
 ('ok', 11),
 ('know', 12),
 ('free', 13),
 ('got', 14),
 ('like', 15),
 ('want', 16),
 ('time', 17),
 ('day', 18),
 ('love', 19),
 ('good', 20)]

In [13]:
X_sequences = tokenizer.texts_to_sequences(X)

In [14]:
X_sequences[0]

[5, 3010, 273, 539, 567, 943, 44, 68, 332, 779, 90, 2104, 944, 14, 3011, 66]

In [15]:
max_length = max(
    len(seq)
    for seq in X_sequences
)

max_length

80

In [16]:
X_padded = pad_sequences(
    X_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

In [17]:
X_padded.shape

(5169, 80)

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_padded,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (3618, 80)
Validation: (775, 80)
Test: (776, 80)


In [19]:
import pickle

with open("../models/rnn_tokenizer.pkl","wb") as f:

    pickle.dump(tokenizer,f)

In [20]:
import os

os.listdir("../models")

['logistic_regression.pkl',
 'naive_bayes.pkl',
 'random_forest.pkl',
 'rnn_tokenizer.pkl',
 'tfidf_vectorizer.pkl']

In [21]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense
)

In [22]:
vocab_size = len(tokenizer.word_index) + 1

print(vocab_size)

7070


In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense
)

rnn_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=64
    ),

    SimpleRNN(64),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

In [24]:
Embedding(input_dim=vocab_size,output_dim=64)

<Embedding name=embedding_1, built=False>

In [25]:
rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [26]:
class_weight = {
    0: 1.0,
    1: 6.9
}

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [41]:
history = rnn_model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=10,

    batch_size=32,

    class_weight=class_weight
)

Epoch 1/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9221 - loss: 0.3899 - val_accuracy: 0.9639 - val_loss: 0.1572
Epoch 2/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9842 - loss: 0.1692 - val_accuracy: 0.9523 - val_loss: 0.2542
Epoch 3/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9873 - loss: 0.0906 - val_accuracy: 0.9161 - val_loss: 0.3252
Epoch 4/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9889 - loss: 0.1044 - val_accuracy: 0.9626 - val_loss: 0.1355
Epoch 5/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9942 - loss: 0.0710 - val_accuracy: 0.9574 - val_loss: 0.2060
Epoch 6/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9701 - loss: 0.2060 - val_accuracy: 0.9471 - val_loss: 0.1925
Epoch 7/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9917 - loss: 0.1233 - val_accuracy: 0.9574 - val_loss: 0.1904
Epoch 8/10
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9099 - loss: 0.5493 - val_accuracy: 0.

In [44]:
test_loss, test_accuracy = rnn_model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", test_accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9601 - loss: 0.1523
Test Accuracy: 0.9600515365600586


In [45]:
y_pred_prob = rnn_model.predict(X_test)

import numpy as np

y_pred = (y_pred_prob > 0.5).astype(int)

y_pred = y_pred.flatten()

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [47]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.9600515463917526
Precision: 0.8383838383838383
Recall   : 0.8469387755102041
F1 Score : 0.8426395939086294


In [48]:
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       678
           1       0.84      0.85      0.84        98

    accuracy                           0.96       776
   macro avg       0.91      0.91      0.91       776
weighted avg       0.96      0.96      0.96       776



In [49]:
rnn_model.save("../models/rnn_model.keras")

In [50]:
import os

os.listdir("../models")

['logistic_regression.pkl',
 'naive_bayes.pkl',
 'random_forest.pkl',
 'rnn_model.keras',
 'rnn_tokenizer.pkl',
 'tfidf_vectorizer.pkl']

In [54]:
rnn_results = {
    "Model": "RNN",
    "Accuracy": rnn_accuracy,
    "Precision": rnn_precision,
    "Recall": rnn_recall,
    "F1": rnn_f1
}

rnn_results_df 

,Model,Accuracy,Precision,Recall,F1
0,RNN,0.960052,0.838384,0.846939,0.84264


Final Model Comparison

In [57]:
comparison_df = pd.DataFrame({

    "Model": [
        "Naive Bayes",
        "Logistic Regression",
        "Random Forest",
        "RNN"
    ],

    "Accuracy": [
        0.9665,
        0.9510,
        0.9716,
        0.9601
    ],

    "Precision": [
        1.0000,
        0.9839,
        0.9872,
        0.8384
    ],

    "Recall": [
        0.7347,
        0.6224,
        0.7857,
        0.8469
    ],

    "F1": [
        0.8471,
        0.7625,
        0.8750,
        0.8426
    ]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.9665,1.0000,0.7347,0.8471
1,Logistic Regression,0.9510,0.9839,0.6224,0.7625
2,Random Forest,0.9716,0.9872,0.7857,0.8750
3,RNN,0.9601,0.8384,0.8469,0.8426


In [63]:
comparison_display = comparison_df.copy()

for col in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]:
    comparison_display[col] = (
        comparison_display[col] * 100
    ).round(2)

comparison_display

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,96.65,100.00,73.47,84.71
1,Logistic Regression,95.10,98.39,62.24,76.25
2,Random Forest,97.16,98.72,78.57,87.50
3,RNN,96.01,83.84,84.69,84.26


Sort by F1 Score

In [73]:
comparison_df.sort_values(
    by="F1",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1
2,Random Forest,0.9716,0.9872,0.7857,0.8750
0,Naive Bayes,0.9665,1.0000,0.7347,0.8471
3,RNN,0.9601,0.8384,0.8469,0.8426
1,Logistic Regression,0.9510,0.9839,0.6224,0.7625


In [72]:
comparison_df.to_csv("../dataset/model_comparison.csv",index=False)

In [71]:
#Best model based on F1 score

best_model = comparison_df.loc[
    comparison_df["F1"].idxmax()
]

best_model

Model        Random Forest
Accuracy            0.9716
Precision           0.9872
Recall              0.7857
F1                   0.875
Name: 2, dtype: object

In [ ]:
print(f"Best Model: {best_model['Model']}")

print(f"F1 Score: {best_model['F1']:.4f}")

Best Model: Random Forest
F1 Score: 87.5000
